# Sonnet Generation with GPT-2

This notebook trains and evaluates the `SonnetGPT` model on Google Colab. It clones the repo, installs dependencies, runs training, and saves the generated sonnets.

## 1. Mount Google Drive & Clone Repo

In [ ]:
import os

REPO_URL = 'https://github.com/Lynx-Zhang/DD2424-Project.git'
REPO_DIR = '/content/DD2424-Project'
BRANCH = 'feat/sonnet-generation'

from google.colab import drive
drive.mount('/content/drive')

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch --all
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

%cd {REPO_DIR}

print("Current branch: ", end='')
!git branch --show-current

!mkdir -p predictions
!mkdir -p /content/drive/MyDrive/sonnet_checkpoints
!mkdir -p /content/drive/MyDrive/sonnet_logs

print("\nGPU info:")
!nvidia-smi -L

print("\nEnvironment ready!")

## 2. Install Dependencies

In [ ]:
!pip install -q \
    tqdm==4.58.0 \
    requests==2.25.1 \
    importlib-metadata==3.7.0 \
    filelock==3.0.12 \
    tokenizers==0.20 \
    explainaboard_client==0.0.7 \
    einops==0.8.0 \
    transformers==4.46.3 \
    sacrebleu==2.5.1 \
    scikit-learn

print("Dependencies installed.")

## 3. Verify Data Files

In [ ]:
!echo "Training sonnets:"
!wc -l data/sonnets.txt

!echo "\nHeld-out sonnets (test, first 3 lines only):"
!wc -l data/sonnets_held_out.txt

!echo "\nHeld-out dev sonnets (val prompts, first 3 lines):"
!wc -l data/sonnets_held_out_dev.txt

!echo "\nTrue held-out dev sonnets (val references):"
!wc -l data/TRUE_sonnets_held_out_dev.txt

## 4. Train — `gpt2` (small, fast baseline)

Trains with early stopping based on validation chrF score.  
The best checkpoint is saved to `best_<epochs>-<lr>-sonnet.pt`.

In [ ]:
!python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2 \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 8 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.9 \
    --sonnet_out predictions/generated_sonnets_gpt2.txt

# Backup checkpoint and predictions to Drive
!cp best_20-1e-05-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_gpt2-$(date +%Y%m%d_%H%M%S).pt
!cp predictions/generated_sonnets_gpt2.txt /content/drive/MyDrive/sonnet_logs/generated_sonnets_gpt2-$(date +%Y%m%d_%H%M%S).txt
print("\nBackup done.")

## 5. (Optional) Train — `gpt2-medium` (better quality, slower)

Uses a larger model. Recommended only if you have a GPU with ≥ 15 GB VRAM (e.g., A100 on Colab Pro).

In [ ]:
!python sonnet_generation.py \
    --use_gpu \
    --model_size gpt2-medium \
    --epochs 20 \
    --lr 1e-5 \
    --batch_size 4 \
    --patience 5 \
    --temperature 1.2 \
    --top_p 0.9 \
    --sonnet_out predictions/generated_sonnets_gpt2medium.txt

!cp best_20-1e-05-sonnet.pt /content/drive/MyDrive/sonnet_checkpoints/best_gpt2medium-$(date +%Y%m%d_%H%M%S).pt
!cp predictions/generated_sonnets_gpt2medium.txt /content/drive/MyDrive/sonnet_logs/generated_sonnets_gpt2medium-$(date +%Y%m%d_%H%M%S).txt
print("\nBackup done.")

## 6. Inspect Generated Sonnets

In [ ]:
OUTPUT_FILE = 'predictions/generated_sonnets_gpt2.txt'  # change to gpt2medium if needed

with open(OUTPUT_FILE) as f:
    content = f.read()

print(content[:3000])

## 7. Re-generate from a Saved Checkpoint (without retraining)

Useful if training already finished and you want to regenerate with different sampling parameters.

In [ ]:
# Restore checkpoint from Drive if needed
# !cp /content/drive/MyDrive/sonnet_checkpoints/<your_checkpoint>.pt best_20-1e-05-sonnet.pt

import torch
import sys
sys.path.insert(0, '/content/DD2424-Project')

from sonnet_generation import SonnetGPT, generate_submission_sonnets, add_arguments
from datasets import SonnetsDataset
import argparse

# Mirror the args used during training
args = argparse.Namespace(
    use_gpu=True,
    model_size='gpt2',
    epochs=20,
    lr=1e-5,
    temperature=1.2,
    top_p=0.9,
    held_out_sonnet_path='data/sonnets_held_out.txt',
    sonnet_out='predictions/generated_sonnets_regen.txt',
)
args.filepath = f'{args.epochs}-{args.lr}-sonnet.pt'

generate_submission_sonnets(args)
print("\nRegeneration complete -> predictions/generated_sonnets_regen.txt")

## 8. Download Predictions

In [ ]:
from google.colab import files
files.download('predictions/generated_sonnets_gpt2.txt')